# Exercise: Build a Graph with Search Tools

This notebook gives you **4 versions** of the same challenge.

Goal: build a graph-driven agent flow that can use a search tool and return a useful answer.

You will implement:

0. **Version 0: Mock LLM + Mock Tool** — understand graph mechanics without any API calls
1. **Version 1: Node + Edge** — real Gemini LLM + Tavily search, routing via conditional edges
2. **Version 2: Node + Command** — same but routing via `Command(goto=...)`
3. **Version 3: `create_agent`** — high-level prebuilt ReAct agent

For each version:
- Read the instructions
- Fill in the TODO blocks (Versions 1–3 only)
- Run the test cell at the end of that version

> **Tip:** Start with Version 0 — it is fully implemented and shows exactly how the graph
> mechanics work before you add a real LLM or search API.

## Shared Setup (Run Once)

Instructions:
1. Run this cell first — every other version depends on it.
2. It loads API keys from `.env`, creates the shared LLM (Gemini or OpenAI), and wraps Tavily as a `@tool`.
3. Set `MODEL_PROVIDER = "gemini"` or `MODEL_PROVIDER = "openai"` at the top of the next cell to choose your model.
4. If an import fails, run `uv add langchain-google-genai` or `uv add langchain-openai` in your terminal and restart the kernel.

In [ ]:
import os
import logging
from typing import TypedDict, Literal

from dotenv import load_dotenv

# LangGraph / LangChain imports
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
from langchain.agents import create_agent

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
# Change this to "openai" to use GPT-4o-mini instead of Gemini.
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert TAVILY_API_KEY, "TAVILY_API_KEY not found — check your .env file"

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

# ── Shared Tavily search tool (LangChain @tool wrapper) ───────────────────────
_tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def tavily_search(query: str) -> str:
    """Search on the internet using Tavily and return a concise answer. 
    This tool will return multiple results, try to use it efficiently.
    Don't call this tool too many times for 1 question, 1 to 2 calls per query from user are enough.
    Give a concise query
    
    Args:
        query: A concise query to search on the internet.
    Returns:
        A concise answer to the user's query.
    """
    print("tavily_search called | query=%s", query)
    response = _tavily_client.search(query=query, max_results=3)
    snippets = [r.get("content", "") for r in response.get("results", [])]
    result = "\n\n".join(snippets) if snippets else "No results found."
    print(f"tavily_search done {len(snippets)} snippets returned")
    return result


# ── Test helper ───────────────────────────────────────────────────────────────
def _assert_contains(text: str, expected: str):
    assert expected.lower() in text.lower(), f"Expected '{expected}' in:\n{text}"


logger.info("Setup complete — LLM: %s | Tavily tool ready", llm.model)

## Version 0: Mock LLM + Mock Tool (Graph Mechanics)

### Goal
Understand **how a graph works** before adding any real API calls.

This version uses the same **weather question** theme as Versions 1–3, but replaces:
- The LLM → a city-name detector (`mock_llm`)
- The search tool → a hardcoded weather dictionary (`mock_search`)

Everything is deterministic — no network, no keys, no surprises.

### Graph shape
```
START → router_node → (needs_search?) → search_node → answer_node → END
                                       ↘ direct_node              → END
```

### What to observe as you run it
1. Each `print` statement shows **which node is executing** and **what data it sees**
2. The state dictionary is passed and updated as it travels through the graph
3. The routing function reads state and returns a string node name

> Once you understand this flow, Versions 1–3 are the same pattern — just swap the
> mock functions for a real LLM and real search API.

In [ ]:
# Version 0: Mock LLM + Mock Tool — fully implemented, just run it!

# ── Mock components ───────────────────────────────────────────────────────────

# Simulates what a real weather search API would return.
MOCK_WEATHER_DB = {
    "tokyo":    "Tokyo: 22°C, partly cloudy, humidity 65%, light breeze from the east.",
    "paris":    "Paris: 15°C, overcast with light rain, humidity 80%, wind 20 km/h.",
    "new york": "New York: 18°C, clear skies, humidity 50%, gentle south wind.",
    "london":   "London: 12°C, foggy morning, humidity 90%, calm winds.",
}

# City names that trigger a search — simulates the LLM recognising "needs real-time data".
KNOWN_CITIES = list(MOCK_WEATHER_DB.keys())


def mock_llm(question: str) -> dict:
    """
    Pretends to be an LLM deciding whether a web search is needed.
    In reality it checks whether the question mentions a known city.
    Returns: {"needs_search": bool, "search_query": str}
    """
    q = question.lower()
    for city in KNOWN_CITIES:
        if city in q:
            print(f"  [mock_llm] detected city '{city}' → needs_search=True")
            return {"needs_search": True, "search_query": city}
    print("  [mock_llm] no city detected → needs_search=False")
    return {"needs_search": False, "search_query": ""}


def mock_search(query: str) -> str:
    """
    Pretends to be a weather search API.
    In reality it looks up the city in MOCK_WEATHER_DB.
    """
    result = MOCK_WEATHER_DB.get(query.lower(), "Weather data not available for this location.")
    print(f"  [mock_search] query='{query}' → '{result[:50]}…'")
    return result


# ── Graph State ───────────────────────────────────────────────────────────────

class V0State(TypedDict):
    question:      str
    needs_search:  bool
    search_query:  str
    search_result: str
    answer:        str


# ── Graph Nodes ───────────────────────────────────────────────────────────────

def v0_router_node(state: V0State) -> V0State:
    """Step 1 — ask mock_llm whether we need a search."""
    print(f"\n→ router_node  | question='{state['question']}'")
    decision = mock_llm(state["question"])
    return {
        **state,
        "needs_search": decision["needs_search"],
        "search_query": decision["search_query"],
    }


def v0_search_node(state: V0State) -> V0State:
    """Step 2a — call mock_search with the query chosen by mock_llm."""
    print(f"→ search_node  | query='{state['search_query']}'")
    result = mock_search(state["search_query"])
    return {**state, "search_result": result}


def v0_answer_node(state: V0State) -> V0State:
    """Step 3a — turn the raw search result into a final answer."""
    print("→ answer_node  | building answer from search result")
    answer = f"Based on search: {state['search_result']}"
    return {**state, "answer": answer}


def v0_direct_node(state: V0State) -> V0State:
    """Step 2b — no search needed, give a generic reply."""
    print("→ direct_node  | no search needed")
    return {**state, "answer": "I don't have specific information about that topic."}


# ── Conditional Routing ───────────────────────────────────────────────────────

def v0_route(state: V0State) -> Literal["search_node", "direct_node"]:
    """Reads state["needs_search"] and returns the name of the next node."""
    target = "search_node" if state["needs_search"] else "direct_node"
    print(f"  [v0_route] needs_search={state['needs_search']} → '{target}'")
    return target


# ── Build Graph ───────────────────────────────────────────────────────────────

v0_builder = StateGraph(V0State)

v0_builder.add_node("router_node", v0_router_node)
v0_builder.add_node("search_node", v0_search_node)
v0_builder.add_node("answer_node", v0_answer_node)
v0_builder.add_node("direct_node", v0_direct_node)

v0_builder.add_edge(START, "router_node")
v0_builder.add_conditional_edges("router_node", v0_route)  # routing happens here
v0_builder.add_edge("search_node", "answer_node")
v0_builder.add_edge("answer_node", END)
v0_builder.add_edge("direct_node", END)

v0_graph = v0_builder.compile()

print("Version 0 graph compiled successfully.")

In [ ]:
# Version 0 self-test — read the printed trace to understand graph execution

EMPTY_V0 = {"needs_search": False, "search_query": "", "search_result": "", "answer": ""}

# Test 1: weather question with a known city → search path
print("=" * 55)
print("Test 1: weather question (should trigger search path)")
print("=" * 55)
v0_result_1 = v0_graph.invoke({"question": "What is the current weather in Tokyo?", **EMPTY_V0})
assert v0_result_1["needs_search"] is True
assert "tokyo" in v0_result_1["answer"].lower(), f"Unexpected answer: {v0_result_1['answer']}"
print(f"\nFinal answer: {v0_result_1['answer']}")

# Test 2: question with no city → direct path (no search)
print("\n" + "=" * 55)
print("Test 2: non-weather question (should take direct path)")
print("=" * 55)
v0_result_2 = v0_graph.invoke({"question": "What is 2 + 2?", **EMPTY_V0})
assert v0_result_2["needs_search"] is False
print(f"\nFinal answer: {v0_result_2['answer']}")

print("\nVersion 0 tests passed ✓")
print("\nNow look at the printed trace above — that is the graph executing node by node.")

## Version 1: Node + Edge (Gemini + Tavily)

### Instructions
- Build a graph with three nodes:
  - `decide_node`: call `llm` to decide whether a web search is needed
  - `search_node`: call `tavily_search` and let `llm` summarize the results
  - `no_search_node`: answer directly with `llm`
- Wire routing with `add_conditional_edges` + a separate routing function.

### Graph shape
```
START → decide_node → (should_search?) → search_node    → END
                                       ↘ no_search_node → END
```

### Expected behavior
- LLM decides if the question needs live search.
- If yes → Tavily fetches snippets → LLM summarizes into `answer`.
- If no → LLM answers directly into `answer`.

In [ ]:
# TODO: Complete Version 1 implementation

class V1State(TypedDict):
    question: str
    should_search: bool
    answer: str


def v1_decide_node(state: V1State) -> V1State:
    # TODO:
    # 1) Build a yes/no prompt asking the LLM if web search is needed
    # 2) Call llm.invoke(prompt) and parse "yes"/"no" from response.content
    # 3) Return {**state, "should_search": <bool>}
    raise NotImplementedError("Implement v1_decide_node")


def v1_search_node(state: V1State) -> V1State:
    # TODO:
    # 1) Call tavily_search.invoke(state["question"]) to get snippets
    # 2) Ask llm to summarize the snippets into an answer
    # 3) Return {**state, "answer": summary.content}
    raise NotImplementedError("Implement v1_search_node")


def v1_no_search_node(state: V1State) -> V1State:
    # TODO:
    # 1) Call llm.invoke(state["question"]) for a direct answer
    # 2) Return {**state, "answer": response.content}
    raise NotImplementedError("Implement v1_no_search_node")


def v1_route(state: V1State) -> Literal["search_node", "no_search_node"]:
    # TODO: return "search_node" if state["should_search"] else "no_search_node"
    raise NotImplementedError("Implement v1_route")


# Build graph (node + edge style)
v1_builder = StateGraph(V1State)
v1_builder.add_node("decide_node", v1_decide_node)
v1_builder.add_node("search_node", v1_search_node)
v1_builder.add_node("no_search_node", v1_no_search_node)

v1_builder.add_edge(START, "decide_node")
# TODO: Add conditional edge from decide_node using v1_route
#       v1_builder.add_conditional_edges("decide_node", v1_route)
# TODO: Add edges from terminal nodes to END
#       v1_builder.add_edge("search_node", END)
#       v1_builder.add_edge("no_search_node", END)

# TODO: compile graph into v1_graph
#       v1_graph = v1_builder.compile()
v1_graph = None

# ── Visualize (auto-renders once you compile above) ───────────────────────────
if v1_graph is not None:
    from IPython.display import Image, display
    display(Image(v1_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Version 1 self-test

assert v1_graph is not None, "v1_graph is None — compile your graph first."

v1_result = v1_graph.invoke({
    "question": "What is the current weather in Tokyo?",
    "should_search": False,
    "answer": "",
})
assert "answer" in v1_result, "Version 1 result must include 'answer'."
assert len(v1_result["answer"]) > 20, f"Answer seems too short: {v1_result['answer']}"

print("Answer:", v1_result["answer"][:300])
print("\nVersion 1 test passed ✓")

## Version 2: Node + Command (Gemini + Tavily)

### Instructions
- Build a graph where routing happens **inside the node** using `Command`.
- The router node calls `llm` to decide search/no-search, then returns:
  ```python
  Command(update={...}, goto="search_node")   # or "no_search_node"
  ```
- No separate routing function needed — the node itself controls flow.

### Difference from Version 1

| | Version 1 | Version 2 |
|---|---|---|
| Routing | `add_conditional_edges` + route fn | `Command(goto=...)` inside node |
| Router returns | `dict` (state update only) | `Command` (state update + next node) |
| Edge wiring | Route function required | No route function needed |

### Graph shape
```
START → router_node ──Command(goto)──► search_node    → END
                                     ↘ no_search_node → END
```

In [ ]:
# TODO: Complete Version 2 implementation

class V2State(TypedDict):
    question: str
    answer: str


def v2_router_node(state: V2State) -> Command[Literal["search_node", "no_search_node"]]:
    # TODO:
    # 1) Build a yes/no prompt and call llm.invoke(prompt)
    # 2) Parse "yes"/"no" from response.content → pick goto target
    # 3) Return Command(update={"question": state["question"], "answer": ""}, goto=<target>)
    raise NotImplementedError("Implement v2_router_node")


def v2_search_node(state: V2State) -> V2State:
    # TODO:
    # 1) Call tavily_search.invoke(state["question"])
    # 2) Summarize snippets with llm
    # 3) Return {**state, "answer": summary.content}
    raise NotImplementedError("Implement v2_search_node")


def v2_no_search_node(state: V2State) -> V2State:
    # TODO:
    # 1) Call llm.invoke(state["question"])
    # 2) Return {**state, "answer": response.content}
    raise NotImplementedError("Implement v2_no_search_node")


v2_builder = StateGraph(V2State)
v2_builder.add_node("router_node", v2_router_node)
v2_builder.add_node("search_node", v2_search_node)
v2_builder.add_node("no_search_node", v2_no_search_node)

v2_builder.add_edge(START, "router_node")
v2_builder.add_edge("search_node", END)
v2_builder.add_edge("no_search_node", END)

# TODO: compile graph into v2_graph
#       v2_graph = v2_builder.compile()
v2_graph = None

# ── Visualize (auto-renders once you compile above) ───────────────────────────
if v2_graph is not None:
    from IPython.display import Image, display
    display(Image(v2_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Version 2 self-test

assert v2_graph is not None, "v2_graph is None — compile your graph first."

v2_result = v2_graph.invoke({
    "question": "What is the current weather in Paris?",
    "answer": "",
})
assert "answer" in v2_result, "Version 2 result must include 'answer'."
assert len(v2_result["answer"]) > 20, f"Answer seems too short: {v2_result['answer']}"

print("Answer:", v2_result["answer"][:300])
print("\nVersion 2 test passed ✓")

## Version 2.5: Manual ReAct Graph (Gemini + Tavily)

### What is ReAct?
**ReAct** = **Re**ason + **Act**. The pattern lets an LLM decide, on every step, whether to:
- call a tool and continue reasoning ("Act"), or
- produce a final answer and stop ("Reason → done").

This is exactly what `create_react_agent` does in Version 3 — but here we build it
**from scratch** so you can see every piece.

### Two nodes, one loop

| Node | What it does |
|------|-------------|
| `model_node` | Calls the LLM with all messages so far. If the LLM wants a tool → go to `tools_node`. If it's done → END. |
| `tools_node` | Runs every tool call from the last `AIMessage` and appends `ToolMessage` results back. Then loops to `model_node`. |

### Graph shape
```
START → model_node ──has tool_calls?──► tools_node ──┐
             ▲                                        │
             └────────────────────────────────────────┘
             │
        no tool_calls → END
```

### Key LangChain helpers used
- `SystemMessage` — prepended on every LLM call to set the agent's role and constraints
- `llm.bind_tools([...])` — tells the LLM which tools it can call
- `ToolNode` — executes tool calls from an `AIMessage` automatically
- `tools_condition` — routing function: returns `"tools"` if there are tool calls, else `END`
- `MessagesState` — built-in state with an `add_messages` reducer (appends instead of overwrites)

> **Version 3 preview:** `create_agent(model, tools)` builds exactly this graph for you.
> After studying this version, the one-liner in Version 3 should feel obvious.

In [ ]:
# Version 2.5: Manual ReAct Graph — fully implemented, study and run it!

# ── System prompt ─────────────────────────────────────────────────────────────
# The system prompt shapes the agent's personality and constraints.
# It is prepended to every LLM call so the model always has this context.
from langchain_core.messages import SystemMessage

V25_SYSTEM_PROMPT = (
    "You are a helpful assistant. "
    "Use the search tool to look up information about the user's query."
    "Only call tool one time for 1 query from user, never call it twice or more."
)

# ── Bind tools to the LLM ─────────────────────────────────────────────────────
# This tells Gemini about the tavily_search tool so it can emit tool_calls.
llm_with_tools = llm.bind_tools([tavily_search])


# ── Nodes ─────────────────────────────────────────────────────────────────────

def v25_model_node(state: MessagesState) -> dict:
    """
    Prepends the system prompt, then calls the LLM with the full message history.
    The LLM either:
      - Returns an AIMessage with tool_calls  → graph routes to tools_node
      - Returns an AIMessage with no tool_calls → graph routes to END
    """
    messages = [SystemMessage(content=V25_SYSTEM_PROMPT)] + state["messages"]
    logger.info("v25_model_node | %d messages in history", len(messages))
    response = llm_with_tools.invoke(messages)
    logger.debug("v25_model_node | tool_calls=%s", getattr(response, "tool_calls", []))
    return {"messages": [response]}


# ToolNode reads tool_calls from the last AIMessage, executes each tool,
# and returns a ToolMessage per call — all automatically.
v25_tool_node = ToolNode(tools=[tavily_search])


# ── Build Graph ───────────────────────────────────────────────────────────────

v25_builder = StateGraph(MessagesState)

v25_builder.add_node("model", v25_model_node)
v25_builder.add_node("tools", v25_tool_node)

v25_builder.add_edge(START, "model")

# tools_condition checks the last message:
#   AIMessage with tool_calls  → "tools"
#   AIMessage with no calls    → END
v25_builder.add_conditional_edges("model", tools_condition)

# After tools run, go back to model so it can reason about the results.
v25_builder.add_edge("tools", "model")

v25_graph = v25_builder.compile()

print("Version 2.5 graph compiled.")
print()
print("Execution flow:")
print("  START → model → (tool_calls?) → tools → model → … → END")

# ── Visualize ─────────────────────────────────────────────────────────────────
from IPython.display import Image, display
display(Image(v25_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Version 2.5 self-test

v25_result = v25_graph.invoke(
    {"messages": [HumanMessage(content="What is the current weather in London?")]}
)

assert "messages" in v25_result
last_message = v25_result["messages"][-1]
text = getattr(last_message, "content", str(last_message))
assert len(str(text)) > 20, f"Answer seems too short: {text}"

# Show the full message trace so you can see the ReAct loop in action.
print("── Message trace ──────────────────────────────────────")
for i, msg in enumerate(v25_result["messages"]):
    role = type(msg).__name__
    content_preview = str(getattr(msg, "content", ""))
    tool_calls = getattr(msg, "tool_calls", [])
    if tool_calls:
        print(f"  [{i}] {role} → tool_calls: {[tc['name'] for tc in tool_calls]}")
    else:
        print(f"  [{i}] {role}: {content_preview}...")

print()
print("Final answer:", str(text))
print("\nVersion 2.5 test passed ✓")

## Version 3: `create_agent` (Gemini + Tavily)

### Instructions
- Use `create_agent` from `langchain.agents` — the highest-level abstraction.
- Pass the shared `llm` (Gemini) and `[tavily_search]` as tools.
- The agent will automatically run a **ReAct loop**: Reason → Act → Observe → Reason…

### What changes from Versions 1 and 2
- You write almost no graph code.
- The agent can call tools **multiple times** in one run.
- You interact via `messages` instead of a custom state dict.

### ReAct flow (managed internally)
```
[HumanMessage] → LLM → tool_call? → [Tavily] → LLM → tool_call? → … → [AIMessage]
```

In [ ]:
# TODO: Complete Version 3 implementation

# TODO:
# 1) Call create_agent(model=llm, tools=[tavily_search])
# 2) Save the returned agent to v3_agent
#
# That's it — create_agent handles the full ReAct loop for you.

v3_agent = None

In [ ]:
# Version 3 self-test

assert v3_agent is not None, "v3_agent is None — create it first."

v3_result = v3_agent.invoke(
    {"messages": [HumanMessage(content="What is the current weather in New York City?")]}
)

assert "messages" in v3_result, "Expected 'messages' key in agent output."
last_message = v3_result["messages"][-1]
text = getattr(last_message, "content", str(last_message))
assert len(str(text)) > 20, f"Answer seems too short: {text}"

print("Answer:", str(text)[:400])
print("\nVersion 3 test passed ✓")

## Wrap-up

After all tests pass, you should understand five patterns:

| Version | LLM | Search | Routing mechanism |
|---------|-----|--------|-------------------|
| 0 | Mock (keyword match) | Mock (dict lookup) | `add_conditional_edges` |
| 1 | Gemini (`llm`) | Tavily (`tavily_search`) | `add_conditional_edges` |
| 2 | Gemini (`llm`) | Tavily (`tavily_search`) | `Command(goto=...)` |
| 2.5 | Gemini (`llm`) | Tavily (`tavily_search`) | Manual ReAct loop (`ToolNode` + `tools_condition`) |
| 3 | Gemini (`llm`) | Tavily (`tavily_search`) | Automatic ReAct loop (`create_agent`) |

### Optional extensions
- Add a calculator `@tool` and update your routing to choose between search and compute
- Give the agent a system prompt via `create_react_agent(..., prompt="You are...")`
- Try streaming: `for chunk in v3_agent.stream(...): print(chunk)`
- Add memory with `MemorySaver` and observe multi-turn conversation